# SPA V9 – Colab T4 (mit CUDA Kernels!)
### Neu in V9:
- ✅ Custom CUDA Kernel V5 (Sparse Attention mit Shared Memory Tiling)
- ✅ Custom CUDA Kernel V7 (Fused Value Aggregation)
- ✅ Automatischer Fallback auf PyTorch wenn Build fehlschlägt
- ✅ Kernel Speedtest nach dem Build
- ✅ Tau Overtrain-Grenze (~73.45) im Plot
- ✅ Alle V8 Features (Pheromone pro Layer, Auto-Tau, Explorer Tokens, Decay)
- ✅ Checkpoint als .pth (safetensors funktioniert nicht beim Neuladen!)

In [1]:
# GPU Check
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import os

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Verfuegbar: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Verwende: {device}')

PyTorch Version: 2.10.0+cu128
CUDA Verfuegbar: True
GPU: Tesla T4
VRAM: 15.6 GB
Verwende: cuda


In [3]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.9 MB/s eta 0:00:00


In [9]:
from torch.utils.cpp_extension import load_inline
import os

CUDA_V5_SRC = """
#include <torch/extension.h>
#include <cuda_runtime.h>

__global__ void sparse_dot_product_tiled_kernel(
    float* logits, const float* q, const float* k,
    const int64_t* topk_indices,
    const int B, const int H, const int T, const int D, const int K)
{
    int b = blockIdx.z;
    int h = blockIdx.y;
    int t_query = blockIdx.x * blockDim.x + threadIdx.x;
    if (t_query < T) {
        extern __shared__ float s_query[];
        for (int d = 0; d < D; d++) {
            s_query[threadIdx.x * D + d] = q[((b * H + h) * T + t_query) * D + d];
        }
        __syncthreads();
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk_indices[b * T * K + t_query * K + k_idx];
            float score = 0.0f;
            for (int d = 0; d < D; d++) {
                score += s_query[threadIdx.x * D + d] * k[((b * H + h) * T + t_key) * D + d];
            }
            logits[((b * H + h) * T + t_query) * K + k_idx] = score;
        }
    }
}

torch::Tensor sparse_attention_v5_tiled(
    torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices)
{
    auto B = q.size(0); auto H = q.size(1);
    auto T = q.size(2); auto D = q.size(3);
    auto K = topk_indices.size(2);
    auto logits = torch::empty({B, H, T, K}, q.options());
    const int threads = 128;
    const dim3 blocks((T + threads - 1) / threads, H, B);
    // Shared Mem: threads*D*float. D=48,threads=128 -> 24KB (T4 hat 48KB -> ok)
    sparse_dot_product_tiled_kernel<<<blocks, threads, threads * D * sizeof(float)>>>(
        logits.data_ptr<float>(), q.data_ptr<float>(), k.data_ptr<float>(),
        topk_indices.data_ptr<int64_t>(), B, H, T, D, K);
    return logits;
}
"""

CUDA_V7_SRC = """
#include <torch/extension.h>

__global__ void fused_value_aggregation_v7_kernel(
    float* out, const float* attn, const float* v,
    const int64_t* topk,
    const int B, const int H, const int T, const int D, const int K)
{
    int t_query = blockIdx.x * blockDim.x + threadIdx.x;
    int d       = blockIdx.y * blockDim.y + threadIdx.y;
    int bh      = blockIdx.z;
    if (t_query < T && d < D) {
        int b = bh / H; int h = bh % H;
        float sum = 0.0f;
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk[b * T * K + t_query * K + k_idx];
            sum += attn[((b * H + h) * T + t_query) * K + k_idx]
                 * v[((b * H + h) * T + t_key) * D + d];
        }
        out[((b * H + h) * T + t_query) * D + d] = sum;
    }
}

torch::Tensor sparse_value_aggregation_v7(
    torch::Tensor attn, torch::Tensor v, torch::Tensor topk)
{
    auto B = v.size(0); auto H = v.size(1);
    auto T = v.size(2); auto D = v.size(3);
    auto K = topk.size(2);
    auto out = torch::empty({B, H, T, D}, v.options());
    dim3 threads(16, 16);
    dim3 blocks((T+threads.x-1)/threads.x, (D+threads.y-1)/threads.y, B*H);
    fused_value_aggregation_v7_kernel<<<blocks, threads>>>(
        out.data_ptr<float>(), attn.data_ptr<float>(), v.data_ptr<float>(),
        topk.data_ptr<int64_t>(), B, H, T, D, K);
    return out;
}
"""

spa_ops_v5 = None
spa_ops_v7 = None

if torch.cuda.is_available():
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
    if 'CUDA_HOME' not in os.environ:
        os.environ['CUDA_HOME'] = '/usr/local/cuda'

    cuda_flags = ['-O3', '-lineinfo', '-gencode=arch=compute_75,code=sm_75']
    print('Kompiliere SPA V9 CUDA Kernels fuer T4 (sm_75)...')

    build_dir = os.path.join(os.getcwd(), 'cuda_extension_build')
    os.makedirs(build_dir, exist_ok=True)

    spa_v5_build_dir = os.path.join(build_dir, 'spa_v5_sm75_build')
    os.makedirs(spa_v5_build_dir, exist_ok=True)
    spa_v7_build_dir = os.path.join(build_dir, 'spa_v7_sm75_build')
    os.makedirs(spa_v7_build_dir, exist_ok=True)

    spa_ops_v5 = load_inline(
        name='spa_v5_sm75',
        cpp_sources='torch::Tensor sparse_attention_v5_tiled(torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices);',
        cuda_sources=CUDA_V5_SRC,
        functions=['sparse_attention_v5_tiled'],
        with_cuda=True,
        extra_cuda_cflags=cuda_flags,
        verbose=True,
        build_directory=spa_v5_build_dir
    )
    spa_ops_v7 = load_inline(
        name='spa_v7_sm75',
        cpp_sources='torch::Tensor sparse_value_aggregation_v7(torch::Tensor attn, torch::Tensor v, torch::Tensor topk);',
        cuda_sources=CUDA_V7_SRC,
        functions=['sparse_value_aggregation_v7'],
        with_cuda=True,
        extra_cuda_cflags=cuda_flags,
        verbose=True,
        build_directory=spa_v7_build_dir
    )
    print('CUDA Kernels aktiv! (V5 Tiled Attention + V7 Fused Value Aggregation)')
else:
    print('Kein CUDA -> PyTorch Fallback aktiv')

Kompiliere SPA V9 CUDA Kernels fuer T4 (sm_75)...
CUDA Kernels aktiv! (V5 Tiled Attention + V7 Fused Value Aggregation)


In [10]:
# ===================== KERNEL SPEEDTEST =====================

if spa_ops_v5 is not None and spa_ops_v7 is not None:
    print('=== CUDA Kernel Speedtest ===')
    B, H, T, D, K = 4, 8, 256, 48, 48
    q_t  = torch.randn(B, H, T, D, device=device)
    k_t  = torch.randn(B, H, T, D, device=device)
    idx_t = torch.randint(0, T, (B, T, K), device=device)

    # Warmup
    for _ in range(5):
        _ = spa_ops_v5.sparse_attention_v5_tiled(q_t, k_t, idx_t)
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(100):
        logits_cuda = spa_ops_v5.sparse_attention_v5_tiled(q_t, k_t, idx_t)
    torch.cuda.synchronize()
    t_cuda = (time.time() - t0) / 100 * 1000

    def pytorch_sparse_attn(q, k, topk_indices):
        B2, H2, T2, D2 = q.shape
        K2 = topk_indices.size(-1)
        b_idx = torch.arange(B2, device=q.device)[:, None, None, None]
        h_idx = torch.arange(H2, device=q.device)[None, :, None, None]
        t_idx = topk_indices[:, None, :, :].expand(B2, H2, T2, K2)
        gathered_k = k[b_idx, h_idx, t_idx, :]
        return (q.unsqueeze(3) * gathered_k).sum(dim=-1)

    t0 = time.time()
    for _ in range(100):
        logits_pt = pytorch_sparse_attn(q_t, k_t, idx_t)
    torch.cuda.synchronize()
    t_pt = (time.time() - t0) / 100 * 1000

    print(f'CUDA Kernel V5:   {t_cuda:.2f} ms')
    print(f'PyTorch Fallback: {t_pt:.2f} ms')
    print(f'Speedup: {t_pt/t_cuda:.1f}x')
    max_diff = (logits_cuda - logits_pt).abs().max().item()
    print(f'Max Abweichung: {max_diff:.2e} ({"OK" if max_diff < 1e-4 else "CHECK!"})')
else:
    print('Kein Speedtest moeglich – CUDA Kernels nicht verfuegbar')

=== CUDA Kernel Speedtest ===
CUDA Kernel V5:   1.85 ms
PyTorch Fallback: 4.60 ms
Speedup: 2.5x
Max Abweichung: 9.54e-06 (OK)


In [11]:
# ===================== HELPER FUNCTIONS =====================
# Automatisch CUDA Kernel oder PyTorch Fallback

def _topk_gather(values, topk_indices):
    B, H, T, D = values.shape
    K = topk_indices.size(-1)
    b_idx = torch.arange(B, device=values.device)[:, None, None, None]
    h_idx = torch.arange(H, device=values.device)[None, :, None, None]
    t_idx = topk_indices[:, None, :, :].expand(B, H, T, K)
    return values[b_idx, h_idx, t_idx, :]

def sparse_attention_impl(q, k, topk_indices):
    """CUDA Kernel V5 wenn verfuegbar, sonst PyTorch."""
    if spa_ops_v5 is not None and q.is_cuda:
        # Kernel erwartet [B, T, K] ohne H
        idx = topk_indices if topk_indices.dim() == 3 else topk_indices[:, 0]
        return spa_ops_v5.sparse_attention_v5_tiled(q, k, idx)
    gathered_k = _topk_gather(k, topk_indices)
    return (q.unsqueeze(3) * gathered_k).sum(dim=-1)

def sparse_value_aggregation_impl(attn, v, topk_indices):
    """CUDA Kernel V7 wenn verfuegbar, sonst PyTorch."""
    if spa_ops_v7 is not None and v.is_cuda:
        idx = topk_indices if topk_indices.dim() == 3 else topk_indices[:, 0]
        return spa_ops_v7.sparse_value_aggregation_v7(attn, v, idx)
    gathered_v = _topk_gather(v, topk_indices)
    return (attn.unsqueeze(-1) * gathered_v).sum(dim=-2)

def safe_topk_indices(scores, k):
    k = min(k, scores.size(-1))
    if k <= 0:
        return torch.zeros(*scores.shape[:-1], 1, dtype=torch.long, device=scores.device)
    return torch.topk(scores, k=k, dim=-1).indices

kernel_mode = 'CUDA V5+V7' if (spa_ops_v5 and spa_ops_v7) else 'PyTorch Fallback'
print(f'Attention Implementation: {kernel_mode}')

Attention Implementation: CUDA V5+V7


In [12]:
# ===================== AUTO-TAU =====================

def tau_regularization_unweighted(model, target=50.0):
    tau = model.current_tau()
    return ((tau - target) / 99.0).pow(2).mean()

def tau_reg_auto_weight(base_weight, ce_loss, reg_unweighted,
                        target_ratio=0.02, min_factor=0.25, max_factor=4.0):
    if reg_unweighted.item() < 1e-8:
        return base_weight
    desired = target_ratio * ce_loss.item()
    current = base_weight * reg_unweighted.item()
    factor  = desired / (current + 1e-8)
    factor  = max(min_factor, min(max_factor, factor))
    return base_weight * factor


# ===================== PHEROMONE UPDATE =====================

def apply_pheromone_update(pheromone_layer, pheromone_decay, pheromone_alpha,
                           attn, combined_tau_idx, T, n_heads):
    with torch.no_grad():
        pheromone = pheromone_layer[:, :T, :T]
        attn_mean = attn.mean(dim=0)
        idx = combined_tau_idx[0].unsqueeze(0).expand(n_heads, -1, -1)
        deposit = torch.zeros_like(pheromone)
        deposit.scatter_add_(dim=-1, index=idx, src=attn_mean)
        pheromone *= (1.0 - pheromone_decay)
        pheromone += deposit
        pheromone.clamp_(0.0, 5.0)

print('Auto-Tau und Pheromone bereit!')

Auto-Tau und Pheromone bereit!


In [13]:
# ===================== SPA V9 MODELL =====================

class SPA_V9_Model(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8, n_layers=6,
                 k=42, max_seq_len=512, local_k=12, explore_k=6,
                 tau_min=1.0, tau_max=100.0, tau_init=40.0,
                 dropout=0.1, pheromone_decay=0.05):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim       = embed_dim
        self.n_heads         = num_heads
        self.head_dim        = embed_dim // num_heads
        self.k               = k
        self.n_layers        = n_layers
        self.max_seq_len     = max_seq_len
        self.local_k         = min(local_k, k)
        self.explore_k       = max(0, min(explore_k, k - self.local_k))
        self.pheromone_decay = pheromone_decay
        self.pheromone_alpha = 0.15
        self.tau_min         = tau_min
        self.tau_max         = tau_max

        self.token_emb  = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb    = nn.Embedding(max_seq_len, embed_dim)

        for i in range(n_layers):
            self.register_buffer(f'pheromone_{i}',
                torch.zeros(self.n_heads, max_seq_len, max_seq_len))

        eps        = 1e-4
        init_ratio = (tau_init - tau_min) / (tau_max - tau_min)
        init_ratio = min(max(init_ratio, eps), 1.0 - eps)
        self.log_tau = nn.Parameter(
            torch.full((1, 1, k), math.log(init_ratio / (1.0 - init_ratio))))

        self.global_router = nn.Linear(embed_dim, max_seq_len)
        self.dropout       = nn.Dropout(dropout)
        self.emb_dropout   = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, 3 * embed_dim),
                nn.Linear(embed_dim, embed_dim),
                nn.LayerNorm(embed_dim),
                nn.Sequential(
                    nn.Linear(embed_dim, 4 * embed_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(4 * embed_dim, embed_dim)
                )
            ]) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def current_tau(self):
        return self.tau_min + (self.tau_max - self.tau_min) * torch.sigmoid(self.log_tau)

    def get_pheromone(self, layer_id):
        return getattr(self, f'pheromone_{layer_id}')

    def _build_sparse_indices(self, x):
        B, T, _ = x.shape
        device   = x.device
        parts    = []

        if self.local_k > 0:
            positions     = torch.arange(T, device=device)
            query_pos     = positions.view(1, T, 1)
            local_offsets = torch.arange(1, self.local_k + 1, device=device).view(1, 1, self.local_k)
            local_idx     = (query_pos - local_offsets).clamp(min=0).expand(B, -1, -1)
            parts.append(local_idx)

        if self.k - self.local_k > 0:
            causal_mask   = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
            router_logits = self.global_router(x)[:, :, :T]
            router_logits = router_logits.masked_fill(causal_mask, float('-inf'))
            gk            = self.k - self.local_k
            learned_k     = max(1, gk // 2)
            learned_idx   = safe_topk_indices(router_logits, learned_k)
            global_parts  = [learned_idx]

            # Explorer Tokens (explore_k=0 -> keine Exploration, kein Rauschen)
            if self.explore_k > 0 and gk - learned_k > 0:
                random_scores = torch.rand(B, T, T, device=device)
                random_scores = random_scores.masked_fill(causal_mask, -1.0)
                explore_idx   = safe_topk_indices(random_scores, gk - learned_k)
                global_parts.append(explore_idx)

            global_idx = torch.cat(global_parts, dim=-1)
            if global_idx.size(-1) > gk:
                global_idx = global_idx[..., :gk]
            elif global_idx.size(-1) < gk:
                pad = torch.zeros(B, T, gk - global_idx.size(-1), device=device, dtype=torch.long)
                global_idx = torch.cat([global_idx, pad], dim=-1)
            parts.append(global_idx)

        combined = torch.cat(parts, dim=-1) if len(parts) > 1 else parts[0]
        return combined.long()

    def forward(self, idx):
        B, T   = idx.shape
        device = idx.device
        x      = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=device))
        x      = self.emb_dropout(x)
        tau_base  = self.current_tau()
        tau_scale = (40.0 / (tau_base + 1e-8)).clamp(0.3, 3.5)

        for layer_id, (ln1, qkv_l, proj, ln2, mlp) in enumerate(self.layers):
            pheromone_buf    = self.get_pheromone(layer_id)
            combined_tau_idx = self._build_sparse_indices(x)
            h   = ln1(x)
            qkv = qkv_l(h).reshape(B, T, 3, self.n_heads, self.head_dim)
            q   = qkv[:, :, 0].transpose(1, 2).contiguous()
            k   = qkv[:, :, 1].transpose(1, 2).contiguous()
            v   = qkv[:, :, 2].transpose(1, 2).contiguous()

            # CUDA Kernel V5 (oder PyTorch Fallback)
            raw_logits = sparse_attention_impl(q, k, combined_tau_idx) * (self.head_dim ** -0.5)

            pheromone          = pheromone_buf[:, :T, :T]
            pheromone_bias     = pheromone.unsqueeze(0).expand(B, -1, -1, -1)
            gathered_pheromone = torch.gather(
                pheromone_bias, dim=-1,
                index=combined_tau_idx.unsqueeze(1).expand(B, self.n_heads, T, -1)
            )
            logits = raw_logits + self.pheromone_alpha * gathered_pheromone
            logits = logits * tau_scale.unsqueeze(1)
            attn   = F.softmax(logits, dim=-1)
            attn   = self.dropout(attn)

            apply_pheromone_update(
                pheromone_buf, self.pheromone_decay, self.pheromone_alpha,
                attn, combined_tau_idx, T, self.n_heads
            )

            # CUDA Kernel V7 (oder PyTorch Fallback)
            out = sparse_value_aggregation_impl(attn, v, combined_tau_idx)
            x   = x + self.dropout(proj(out.transpose(1, 2).reshape(B, T, self.embed_dim)))
            x   = x + mlp(ln2(x))

        return self.head(self.ln_f(x)), tau_base, combined_tau_idx

print(f'SPA V9 Modell bereit! ({kernel_mode})')

SPA V9 Modell bereit! (CUDA V5+V7)


In [14]:
# Dataset laden
import requests

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
r   = requests.get(url)
with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(r.text)
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars      = sorted(list(set(text)))
VOCAB_SIZE = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}
itos       = {i: ch for i, ch in enumerate(chars)}
encode     = lambda s: [stoi[c] for c in s]
decode     = lambda l: ''.join([itos[i] for i in l])
data       = torch.tensor(encode(text), dtype=torch.long)
print(f'Vocab Size: {VOCAB_SIZE} | Dataset: {len(data)} Zeichen')

Vocab Size: 65 | Dataset: 1115394 Zeichen


In [15]:
# ===================== MODELL KONFIGURATION =====================
# Explorer Tokens:
#   explore_k=6  -> kreativ, erfindet neue Woerter (z.B. GHAMNGBROKE)
#   explore_k=0  -> fokussiert, bleibt bei gelernten Mustern

model = SPA_V9_Model(
    vocab_size=VOCAB_SIZE,
    embed_dim=384,
    num_heads=8,
    n_layers=6,
    k=48,
    max_seq_len=512,
    local_k=12,
    explore_k=6,      # <- 0 fuer kein Explorer
    tau_init=40.0,
    dropout=0.1,
    pheromone_decay=0.05
).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Modellparameter: {total_params:.2f}M')
if torch.cuda.is_available():
    print(f'GPU Speicher: {torch.cuda.memory_allocated() / 1e6:.1f} MB')

Modellparameter: 11.09M
GPU Speicher: 101.5 MB


In [16]:
# ===================== TRAINING SETUP =====================

BATCH_SIZE            = 32
BLOCK_SIZE            = 256
TRAIN_STEPS           = 3500
CHECKPOINT_EVERY      = 1000
TAU_TARGET            = 50.0
TAU_REG_WEIGHT        = 1e-4
TAU_REG_WARMUP_STEPS  = 2000
AUTO_TAU_TUNE         = True
TAU_REG_TARGET_RATIO  = 0.02
TAU_REG_AUTO_MIN_FACTOR = 0.25
TAU_REG_AUTO_MAX_FACTOR = 4.0
TAU_REG_EMA_BETA      = 0.95
tau_reg_weight_ema    = TAU_REG_WEIGHT

n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    xb = torch.stack([d[j:j+BLOCK_SIZE] for j in ix]).to(device)
    yb = torch.stack([d[j+1:j+BLOCK_SIZE+1] for j in ix]).to(device)
    return xb, yb

@torch.no_grad()
def estimate_val_loss(eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for ki in range(eval_iters):
        xb, yb = get_batch('val')
        logits, _, _ = model(xb)
        losses[ki] = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1)).item()
    model.train()
    return losses.mean().item()

optimizer = torch.optim.AdamW([
    {'params': [p for n2, p in model.named_parameters() if 'log_tau' not in n2], 'lr': 3e-4},
    {'params': model.log_tau, 'lr': 1e-3}
], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_STEPS)

print(f'Training Setup bereit: {TRAIN_STEPS} Steps, Batch={BATCH_SIZE}')

Training Setup bereit: 3500 Steps, Batch=32


In [ ]:
# ===================== TRAINING LOOP =====================

model.train()
metrics = {'step': [], 'train_loss': [], 'val_loss': [], 'perplexity': [], 'tau': []}

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

training_start    = time.time()
step_start        = time.time()
tokens_since_last = 0

print(f'Starte SPA V9 Training ({TRAIN_STEPS} Steps, {kernel_mode})...\n')

for step in range(TRAIN_STEPS + 1):
    xb, yb = get_batch('train')
    tokens_since_last += BATCH_SIZE * BLOCK_SIZE

    logits, tau_vals, _ = model(xb)
    ce_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    tau_reg_raw        = tau_regularization_unweighted(model, target=TAU_TARGET)
    tau_reg_weight_auto = tau_reg_auto_weight(
        base_weight=TAU_REG_WEIGHT, ce_loss=ce_loss, reg_unweighted=tau_reg_raw,
        target_ratio=TAU_REG_TARGET_RATIO,
        min_factor=TAU_REG_AUTO_MIN_FACTOR, max_factor=TAU_REG_AUTO_MAX_FACTOR,
    ) if AUTO_TAU_TUNE else TAU_REG_WEIGHT

    tau_reg_weight_ema = TAU_REG_EMA_BETA * tau_reg_weight_ema + (1.0 - TAU_REG_EMA_BETA) * tau_reg_weight_auto
    tau_reg_weight_cur = min(1.0, step / max(1, TAU_REG_WARMUP_STEPS)) * tau_reg_weight_ema
    loss = ce_loss + tau_reg_weight_cur * tau_reg_raw

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    if step % 500 == 0:
        val_loss    = estimate_val_loss()
        ppl         = math.exp(val_loss)
        current_tau = model.current_tau().mean().item()
        elapsed     = time.time() - training_start
        tok_per_sec = tokens_since_last / (time.time() - step_start + 1e-8)
        step_start  = time.time()
        tokens_since_last = 0

        metrics['step'].append(step)
        metrics['train_loss'].append(ce_loss.item())
        metrics['val_loss'].append(val_loss)
        metrics['perplexity'].append(ppl)
        metrics['tau'].append(current_tau)

        print(f'Step {step:5d} | Loss: {ce_loss.item():.4f} | Val: {val_loss:.4f} | '
              f'PPL: {ppl:.2f} | Tau: {current_tau:.1f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e} | '
              f'Tok/s: {tok_per_sec:.0f} | Zeit: {elapsed:.0f}s')

    if step % CHECKPOINT_EVERY == 0 and step > 0:
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics,
        }, f'spa_v9_checkpoint_step{step}.pth')
        print(f'  -> Checkpoint gespeichert: spa_v9_checkpoint_step{step}.pth')

total_time = time.time() - training_start
gpu_mem    = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0
print(f'\n{"="*55}')
print(f'SPA V9 FINAL | Val Loss: {metrics["val_loss"][-1]:.4f} | PPL: {metrics["perplexity"][-1]:.2f}')
print(f'Zeit: {total_time/60:.1f} min | GPU Peak: {gpu_mem:.1f} MB | {kernel_mode}')
print(f'{"="*55}')

Starte SPA V9 Training (3500 Steps, CUDA V5+V7)...

Step     0 | Loss: 4.3510 | Val: 3.7637 | PPL: 43.11 | Tau: 40.0 | LR: 3.00e-04 | Tok/s: 972 | Zeit: 8s
Step   500 | Loss: 2.4007 | Val: 2.4430 | PPL: 11.51 | Tau: 48.7 | LR: 2.85e-04 | Tok/s: 31211 | Zeit: 140s


In [ ]:
# ===================== SPEICHERN =====================
# WICHTIG: .pth verwenden! safetensors funktioniert nicht beim Neuladen.

torch.save({
    'step': TRAIN_STEPS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'metrics': metrics,
}, 'spa_v9_final.pth')
print('Gespeichert: spa_v9_final.pth')

# Optional: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# torch.save(checkpoint, '/content/drive/MyDrive/spa_v9_final.pth')

In [ ]:
# ===================== LADEN =====================

checkpoint   = torch.load('spa_v9_final.pth', map_location=device)
model_loaded = SPA_V9_Model(
    vocab_size=VOCAB_SIZE, embed_dim=384, num_heads=8, n_layers=6,
    k=48, max_seq_len=512, local_k=12, explore_k=6,
    tau_init=40.0, dropout=0.1, pheromone_decay=0.05
).to(device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
metrics    = checkpoint.get('metrics', {})
start_step = checkpoint.get('step', 0)
print(f'SPA V9 geladen! (Step {start_step})')

In [ ]:
# ===================== TEXT GENERIERUNG =====================

def top_p_sampling(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cum > p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0]  = 0
    logits[sorted_indices[remove]] = float('-inf')
    return logits

model.eval()
start_context      = 'ROMEO: To '
GEN_LENGTH         = 600
TEMPERATURE        = 0.65
TOP_P              = 0.9
REPETITION_PENALTY = 1.3
PENALTY_WINDOW     = 0
T_CTX              = model.max_seq_len

if torch.cuda.is_available():
    torch.cuda.empty_cache()

idx = torch.tensor(encode(start_context), dtype=torch.long).unsqueeze(0).to(device)
gen_start = time.time()
print(f'Generiere {GEN_LENGTH} Tokens (Temp: {TEMPERATURE}, T_CTX: {T_CTX})...')

with torch.no_grad():
    for i in range(GEN_LENGTH):
        idx_cond = idx[:, -T_CTX:]
        logits, _, _ = model(idx_cond)
        logits = logits[0, -1, :] / TEMPERATURE
        recent = set(idx[0, -PENALTY_WINDOW:].tolist())
        for ct in recent:
            logits[ct] = logits[ct] / REPETITION_PENALTY if logits[ct] > 0 else logits[ct] * REPETITION_PENALTY
        logits   = top_p_sampling(logits, p=TOP_P)
        probs    = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat((idx, next_idx.unsqueeze(0)), dim=1)

gen_time = time.time() - gen_start
print(f'Zeit: {gen_time:.1f}s | Tok/s: {GEN_LENGTH/gen_time:.1f}')
print(f'\n--- Ergebnis SPA V9 ({kernel_mode}) ---\n')
print(decode(idx[0].tolist()))

In [ ]:
# ===================== TRAINING KURVE =====================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(metrics['step'], metrics['train_loss'], label='Train', color='tab:red', alpha=0.7)
axes[0].plot(metrics['step'], metrics['val_loss'],   label='Val',   color='tab:blue')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(metrics['step'], metrics['perplexity'], color='tab:green')
axes[1].set_title('Perplexity')

axes[2].plot(metrics['step'], metrics['tau'], color='tab:orange')
axes[2].axhline(y=73.45, color='red', linestyle='--', alpha=0.6, label='Overtrain ~73.45')
axes[2].set_title('Tau Verlauf'); axes[2].legend()

for ax in axes: ax.set_xlabel('Steps')
plt.suptitle(f'SPA V9 Training ({kernel_mode})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ===================== PHEROMONE VISUALISIERUNG =====================
!pip install seaborn -q
import seaborn as sns
import matplotlib.pyplot as plt

def plot_all_pheromone_layers(model, seq_len=128):
    n    = model.n_layers
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    axes = axes.flatten()
    for i in range(n):
        phero = model.get_pheromone(i).detach().cpu().numpy()
        avg   = phero.mean(axis=0)[:seq_len, :seq_len]
        sns.heatmap(avg, cmap='hot', ax=axes[i], cbar=True)
        axes[i].set_title(f'Layer {i}')
    for i in range(n, len(axes)):
        axes[i].set_visible(False)
    plt.suptitle('SPA V9 - Pheromon pro Layer (jetzt getrennt!)', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_all_pheromone_layers(model, seq_len=128)